# **Online preference optimisation for protein design**

### A minimal Colab practical accompanying *Steering Generative Models for Protein Design: Aligning and Conditioning Strategies*

We will align **ProtGPT3-112M** towards a deliberately simple objective: generating longer sequences. The purpose is to make the loop visible:

**current model → two new sequences → reward → preferred sequence → DPO update → repeat**

Online DPO updates model parameters using preferences. It is not a policy-gradient algorithm, but it belongs to the broader family of model-alignment methods discussed alongside reinforcement learning.

> Select **Runtime → Change runtime type → T4 GPU**. This is a classroom demonstration, not a protein-validation pipeline.


## **Learning objectives**

By the end, you should be able to:

1. identify the model, generated sequence, reward, and update in an alignment loop;
2. explain how two scalar rewards become a preference;
3. change the objective by editing one short Python function; and
4. recognise when a model is gaming the reward.


## **1. What makes this online?**

At every training step, the **current** model generates two sequences from the same prompt. We score both and prefer the one with the larger reward. DPO then makes that preference more likely relative to a fixed copy of the original model.

The trainer uses the familiar DPO loss:

$$
\mathcal{L}_{\mathrm{DPO}}=-\log\sigma\left(\beta\left[
\log\frac{\pi_\theta(x_w)}{\pi_{\mathrm{ref}}(x_w)}-
\log\frac{\pi_\theta(x_l)}{\pi_{\mathrm{ref}}(x_l)}
\right]\right),
$$

where $x_w$ and $x_l$ are the preferred and rejected sequences. We will edit the biological objective, while TRL handles padding, token log-probabilities, and the DPO loss.


### **Possible protein rewards**

| Reward | Simple definition | What can go wrong? |
|---|---|---|
| Length | number of residues | the model avoids the end token and hits the generation cap |
| Target length | negative distance from a chosen length | says nothing about fold or function |
| Mean pLDDT | mean structure-prediction confidence | confidence is not stability or activity |
| CLEAN | score for a target EC class | predicted annotation is not demonstrated catalysis |

We begin with length because its behaviour—and its failure mode—are easy to inspect.


## **2. Setup**


In [ ]:
%pip install -q "trl==1.13.0" "transformers>=4.56.2,<5" "peft>=0.15,<1" datasets accelerate


**Mini-exercise.** Find TRL in the installation line. Which part of the loop do you expect this library to implement for us?


In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl.experimental.online_dpo import OnlineDPOConfig, OnlineDPOTrainer

assert torch.cuda.is_available(), "Please select a GPU runtime and run again."


**Mini-exercise.** Classify each import as data handling, model handling, parameter-efficient training, or preference optimisation.


## **3. Load the protein language model**


In [ ]:
model_id = "AI4PD/ProtGPT3-112M"
tokenizer = AutoTokenizer.from_pretrained(
    model_id, add_bos_token=True, add_eos_token=False
)
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16
).cuda()


**Mini-exercise.** Print `model.config.num_hidden_layers` and `hidden_size`. Which quantity describes depth, and which describes the size of each hidden representation?


ProtGPT3 uses the short prompt `1` to request N-to-C generation. The helper below contains only the sampling choices we want students to change.


In [ ]:
amino_acids = set("ACDEFGHIKLMNPQRSTVWY")

def clean(text):
    return "".join(character for character in text if character in amino_acids)

@torch.inference_mode()
def sample(model, number=16, seed=1):
    torch.manual_seed(seed)
    prompt = tokenizer("1", return_tensors="pt").to(model.device)
    output = model.generate(
        **prompt, do_sample=True, temperature=1.0, top_p=0.95,
        max_new_tokens=160, num_return_sequences=number,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    new_tokens = output[:, prompt["input_ids"].shape[1]:]
    texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return [clean(text) for text in texts]


**Mini-exercise.** Change `temperature` to `0.7`, generate again later, and predict whether the sequences should become more or less diverse.


## **4. Inspect the model before alignment**


In [ ]:
before = sample(model, number=16, seed=123)
before_table = pd.DataFrame({"sequence": before})
before_table["length"] = before_table.sequence.str.len()
before_table


**Mini-exercise.** Calculate the mean length. Then count how many sequences reached the 160-residue generation cap.


## **5. Write the reward**

A custom reward function receives the newly generated completions and must return one number per completion. Higher is better.


In [ ]:
def length_reward(completions, **kwargs):
    return [len(clean(sequence)) for sequence in completions]


**Mini-exercise.** Replace the return line with `-abs(len(clean(sequence)) - 100)` to reward a target length of 100 residues. Which sequence gets the best possible score?


In [ ]:
example = before[:2]
print(example[0], "→", length_reward([example[0]])[0])
print(example[1], "→", length_reward([example[1]])[0])
print("Preferred:", max(example, key=len))


**Mini-exercise.** Swap `max` for `min`. Have you changed the model, the reward, or only the way the preference is selected?


## **6. Give the trainer prompts**

Online DPO needs prompts rather than a precomputed preference dataset: the trainer generates fresh competing sequences while the model is changing. Every row contains the same direction prompt because our objective applies to unconditional ProtGPT3 generation.


In [ ]:
prompts = Dataset.from_dict({"prompt": ["1"] * 40})
prompts


**Mini-exercise.** Change one prompt from `1` to `2`. What generation direction does `2` request, and would mixing directions make this comparison easier or harder to interpret?


## **7. Choose the small trainable update**

LoRA freezes the original weights and trains small adapter matrices. This keeps the practical light and leaves a fixed reference policy underneath the adapters.


In [ ]:
lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)


**Mini-exercise.** Change `r` from 8 to 4. What do you expect to happen to the number of trainable parameters?


The settings below deliberately run only a few updates. `beta` controls how strongly DPO compares the changing policy with the fixed reference. The generation cap is also the loophole in our length reward.


In [ ]:
settings = OnlineDPOConfig(
    output_dir="online-dpo-length",
    max_steps=8,
    per_device_train_batch_size=4,
    learning_rate=1e-5,
    beta=0.1,
    max_new_tokens=160,
    temperature=1.0,
    top_p=0.95,
    fp16=True,
    gradient_checkpointing=False,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)


**Mini-exercise.** Increase `max_steps` only after the short run works. Which setting would you change to make hitting the generation cap less attractive?


In [ ]:
trainer = OnlineDPOTrainer(
    model=model,
    reward_funcs=length_reward,
    args=settings,
    processing_class=tokenizer,
    train_dataset=prompts,
    peft_config=lora,
)
trainer.model.print_trainable_parameters()


**Mini-exercise.** Compare trainable parameters with total parameters. Why do we still call this parameter-updating alignment?


## **8. Run online DPO**


In [ ]:
trainer.train()


**Mini-exercise.** In the training output, find the external score and the KL-related metric. Should both always increase?


In [ ]:
history = pd.DataFrame(trainer.state.log_history)
history.filter(regex="step|loss|score|reward|kl").dropna(how="all")


**Mini-exercise.** Plot one numeric column against `step`. Which metric most directly reflects the reward function you wrote?


## **9. Compare fresh samples before and after training**

We reuse the same random seed so that the policy update—not a different requested seed—is the main difference.


In [ ]:
after = sample(trainer.model, number=16, seed=123)
results = pd.DataFrame({
    "policy": ["before"] * len(before) + ["after"] * len(after),
    "sequence": before + after,
})
results["length"] = results.sequence.str.len()
results.groupby("policy").length.agg(["mean", "median", "max"])


**Mini-exercise.** Did the mean length increase? Is the effect also visible in the median, or is it driven by only a few sequences?


In [ ]:
results.boxplot(column="length", by="policy", grid=False)
results.assign(at_cap=results.length >= 160).groupby("policy").at_cap.mean()


**Mini-exercise.** If many aligned sequences reach 160 residues, explain why that is reward hacking rather than evidence of better proteins.


In [ ]:
results.query("policy == 'after'").sort_values("length", ascending=False).head()


**Mini-exercise.** Inspect the longest sequences for repetition or unusual composition. Add one simple diagnostic of your choice.


## **10. Change the scientific question, not the whole notebook**

To optimise another property, replace `length_reward` and pass the new function as `reward_funcs` when creating the trainer.

- **Target length:** return `-abs(length - target)`.
- **pLDDT:** fold each cleaned completion and return mean pLDDT. This is slow, and pLDDT measures predictor confidence—not experimental stability or function.
- **CLEAN:** return a score for a chosen EC class. CLEAN supplies a predicted annotation—not demonstrated catalysis.
- **Multiple objectives:** put metrics on comparable scales before adding them, and always inspect each component separately.

This is a sensible place to use a coding assistant: ask it to replace **only the reward function**, then verify that it returns exactly one finite number for every input sequence and that you can explain what each number means.


In [ ]:
def target_length_reward(completions, target=100, **kwargs):
    return [-abs(len(clean(sequence)) - target) for sequence in completions]


**Mini-exercise.** Use `target_length_reward` in a new short run. Compare its cap fraction with the unbounded length reward. Do not overwrite the first result.


## **Take-home messages**

- The **reward function is the scientific specification**; the trainer is plumbing.
- Online DPO creates preferences from fresh samples produced by the changing policy.
- Higher reward does not automatically mean a better protein. Always look for shortcuts, diversity loss, and disagreement with independent evaluation.
- Short, editable code makes it easier to ask and test one scientific question at a time.

**Sources:** [review preprint](https://arxiv.org/abs/2511.21476), [TRL Online DPO documentation](https://huggingface.co/docs/trl/v1.13.0/en/online_dpo_trainer), [ProtGPT3-112M](https://huggingface.co/AI4PD/ProtGPT3-112M), [DPO](https://arxiv.org/abs/2305.18290), [ESMFold](https://doi.org/10.1126/science.ade2574), and [CLEAN](https://doi.org/10.1126/science.adf2465).
